# Pipeline de Clustering d'Urgence Médicale - 5 Niveaux

**Flux de travail complet pour grouper les témoignages médicaux en 5 niveaux d'urgence**

- 🟢 **Bénin** (Niveau 1) : Symptômes légers, mineurs
- 🟢 **Faible** (Niveau 2) : Inconfort faible, sourd  
- 🟡 **Modéré** (Niveau 3) : Douleur modérée et symptômes courants
- 🟠 **Grave** (Niveau 4) : Symptômes sévères, aigus
- 🔴 **Urgent** (Niveau 5) : Symptômes critiques, urgence

**Étapes du processus :**
1. Chargement et exploration des données
2. Calcul des scores d'urgence avec mots-clés pondérés
3. Attribution des clusters basés sur les niveaux d'urgence
4. Vectorisation des textes avec TF-IDF
5. Réduction des dimensions avec PCA
6. Visualisation avec UMAP
7. Analyse détaillée des clusters
8. Sauvegarde des résultats

## Architecture générale de la pipeline

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                     PIPELINE DE CLUSTERING MÉDICALE                         │
└─────────────────────────────────────────────────────────────────────────────┘

                                   ENTRÉE
                                     │
                                     ▼
                        ┌───────────────────────┐
                        │   Textes bruts        │
                        │  (1011 témoignages)   │
                        └───────────────────────┘
                                     │
                 ┌───────────────────┴───────────────────┐
                 │                                       │
                 ▼                                       ▼
        ┌──────────────────┐               ┌──────────────────────┐
        │   VECTORISATION  │               │  CALCUL DES SCORES   │
        │     (TF-IDF)     │               │   D'URGENCE          │
        │  100 features    │               │  (Mots-clés pondérés)│
        │  Bigrammes       │               │  Score 1.0 → 5.0     │
        │ 1011 × 100       │               │          │           │
        └────────┬─────────┘               └──────────┬───────────┘
                 │                                    │
                 ▼                                    │
        ┌──────────────────┐                         │
        │ RÉDUCTION (PCA)  │◄────────────────────────┘
        │  20 composantes  │
        │   Variance: 39%  │
        │  1011 × 20       │
        └─────────┬────────┘
                  │
                  ▼
        ┌──────────────────────┐
        │  CLASSIFICATION      │
        │  5 niveaux d'urgence │
        │  (Clusters)          │
        │  • Bénin      (0)    │
        │  • Faible     (1)    │
        │  • Modéré     (2)    │
        │  • Grave      (3)    │
        │  • Urgent     (4)    │
        └─────────┬────────────┘
                  │
                  ▼
        ┌──────────────────┐
        │ VISUALISATION    │
        │    (UMAP 2D)     │
        │  + Augmentation  │
        │  du poids urgence│
        │  1011 × 2        │
        └─────────┬────────┘
                  │
      ┌───────────┴────────────┐
      │                        │
      ▼                        ▼
  ┌─────────────┐      ┌──────────────┐
  │  GRAPHIQUE  │      │  FICHIER CSV │
  │  UMAP Scatter│      │   Résultats  │
  │  + Labels    │      │  5 colonnes  │
  └─────────────┘      └──────────────┘
      
        SORTIES : Visualisations + Données étiquetées
```


## 1. Importer les bibliothèques requises

In [ ]:
import csv
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap

# Supprimer les avertissements pour une sortie plus propre
warnings.filterwarnings("ignore")

# Configurer matplotlib pour l'affichage dans le notebook
plt.rcParams['figure.figsize'] = (14, 8)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Toutes les bibliothèques importées avec succès !")

## 2. Charger et explorer les données d'urgence

In [ ]:
# Charger le dataset d'urgence
CHEMIN_DONNEES = os.path.join("data", "unsupervised", "processed", "Student_Dataset_urgency.csv")

ids = []
texts = []

with open(CHEMIN_DONNEES, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) >= 3:
            ids.append(row[0])
            texts.append(row[2])

print(f"✓ Dataset chargé : {len(texts)} témoignages médicaux")
print(f"\n📊 Informations du dataset :")
print(f"   - Total d'échantillons : {len(texts)}")
print(f"   - Premiers 3 échantillons :")
for i, text in enumerate(texts[:3], 1):
    preview = text[:80] + "..." if len(text) > 80 else text
    print(f"   [{i}] {preview}")

## 3. Calculer les scores d'urgence avec mots-clés pondérés

### Définir les niveaux d'urgence avec poids des mots-clés

In [ ]:
# Définir les niveaux d'urgence avec mots-clés pondérés
MOTS_CLES_URGENCE = {
    "benin": {
        "keywords": ["mild", "slight", "minor", "small", "tender", "discomfort", "barely", "almost", "ease"],
        "weight": 1
    },
    "faible": {
        "keywords": ["ache", "soreness", "sore", "dull", "weak", "minimal", "light", "tired"],
        "weight": 2
    },
    "modere": {
        "keywords": ["pain", "fever", "headache", "cough", "throat", "itchy", "rash", "swell", "inflame", "moderate"],
        "weight": 3
    },
    "grave": {
        "keywords": ["severe", "intense", "sharp", "acute", "emergency", "critical", "spread", "worse", "urgent", "throb"],
        "weight": 4
    },
    "urgent": {
        "keywords": ["chest", "pressure", "breathe", "difficulty", "faint", "unconscious", "blood", "emergency", "hospital", "ambulance"],
        "weight": 5
    }
}

# Afficher les mots-clés d'urgence
print("📋 Mots-clés d'urgence par niveau :\n")
for level, data in MOTS_CLES_URGENCE.items():
    weight = data["weight"]
    keywords = data["keywords"]
    print(f"  🟢 Niveau {weight} ({level.upper()}) : {', '.join(keywords)}")

In [ ]:
def calculer_score_urgence(text):
    """
    Calculer le score d'urgence (1-5) pour un texte basé sur les mots-clés pondérés.
    Retourne un float entre 1 (bénin) et 5 (urgent)
    """
    text_lower = text.lower()
    total_score = 0
    total_weight = 0
    
    for level, data in MOTS_CLES_URGENCE.items():
        weight = data["weight"]
        for keyword in data["keywords"]:
            count = text_lower.count(keyword)
            total_score += count * weight
            total_weight += count
    
    if total_weight == 0:
        return 3.0  # Modéré par défaut si aucun mot-clé trouvé
    
    return min(5.0, max(1.0, total_score / total_weight))

# Calculer les scores d'urgence pour tous les textes
print("⏳ Calcul des scores d'urgence pour tous les textes...")
urgency_scores = np.array([calculer_score_urgence(text) for text in texts])

print(f"\n✓ Scores d'urgence calculés :")
print(f"   - Score min : {urgency_scores.min():.2f}")
print(f"   - Score max : {urgency_scores.max():.2f}")
print(f"   - Score moyen : {urgency_scores.mean():.2f}")
print(f"   - Écart-type : {urgency_scores.std():.2f}")

## 4. Attribution des clusters basés sur les niveaux d'urgence

Attribution des clusters basée sur les plages de scores d'urgence :

In [ ]:
# Attribution des clusters basée sur les plages de scores d'urgence
labels = np.zeros(len(texts), dtype=int)

for i, score in enumerate(urgency_scores):
    if score < 1.8:
        labels[i] = 0  # Bénin
    elif score < 2.6:
        labels[i] = 1  # Faible
    elif score < 3.4:
        labels[i] = 2  # Modéré
    elif score < 4.2:
        labels[i] = 3  # Grave
    else:
        labels[i] = 4  # Urgent

# Afficher la distribution des clusters
noms_niveaux_urgence = {0: "🟢 Bénin", 1: "🟢 Faible", 2: "🟡 Modéré", 3: "🟠 Grave", 4: "🔴 Urgent"}
plages_scores = {0: "< 1.8", 1: "1.8-2.6", 2: "2.6-3.4", 3: "3.4-4.2", 4: "≥ 4.2"}

print("📊 Distribution des clusters :\n")
unique, counts = np.unique(labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    nom_niveau = noms_niveaux_urgence[cluster_id]
    plage_scores = plages_scores[cluster_id]
    pourcentage = (count / len(texts)) * 100
    print(f"  {nom_niveau} : {count:4d} échantillons ({pourcentage:5.1f}%) - Plage de score : {plage_scores}")

print(f"\n  Total : {len(texts)} échantillons")

## 5. Vectoriser les textes avec TF-IDF

In [ ]:
# Vectoriser les textes avec TF-IDF
print("⏳ Vectorisation des textes avec TF-IDF...")

vectorizer = TfidfVectorizer(
    max_features=100,      # Limiter à 100 features
    min_df=2,              # Minimum 2 documents
    max_df=0.9,            # Maximum 90% des documents
    ngram_range=(1, 2),    # Unigrammes et bigrammes
    sublinear_tf=True      # Appliquer le scaling sublinéaire tf
)

tfidf_matrix = vectorizer.fit_transform(texts)

print(f"\n✓ Vectorisation TF-IDF terminée :")
print(f"   - Forme de la matrice : {tfidf_matrix.shape}")
print(f"   - Nombre de features : {tfidf_matrix.shape[1]}")
print(f"   - Sparsité : {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")

# Afficher les features d'exemple
feature_names = vectorizer.get_feature_names_out()
print(f"\n📝 Features d'exemple (premiers 20) :")
print(f"   {', '.join(feature_names[:20])}")

## 6. Réduire les dimensions avec PCA

Appliquer PCA pour réduire de 100 dimensions à 20 composantes pour l'efficacité computationnelle et la visualisation.

In [ ]:
# Réduire les dimensions avec PCA
print("⏳ Application de la réduction dimensionnelle PCA...")

pca = PCA(n_components=20)
# Convertir la matrice sparse en dense pour PCA
reduced_data = pca.fit_transform(tfidf_matrix.toarray())

# Normaliser les données réduites
scaler = StandardScaler()
reduced_data_scaled = scaler.fit_transform(reduced_data)

print(f"\n✓ Réduction dimensionnelle PCA terminée :")
print(f"   - Dimensions originales : {tfidf_matrix.shape[1]}")
print(f"   - Dimensions réduites : {reduced_data.shape[1]}")
print(f"   - Variance totale expliquée : {pca.explained_variance_ratio_.sum():.2%}")
print(f"   - Variance par composante (top 5) : {pca.explained_variance_ratio_[:5]}")

# Tracer la variance expliquée cumulée
fig, ax = plt.subplots(figsize=(12, 5))
cumsum = np.cumsum(pca.explained_variance_ratio_)
ax.plot(range(1, len(cumsum) + 1), cumsum, 'bo-', linewidth=2, markersize=8)
ax.axhline(y=0.9, color='r', linestyle='--', label='seuil 90%')
ax.set_xlabel('Nombre de composantes', fontsize=12, fontweight='bold')
ax.set_ylabel('Variance expliquée cumulée', fontsize=12, fontweight='bold')
ax.set_title('PCA - Variance expliquée cumulée', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n   Forme des données réduites : {reduced_data_scaled.shape}")

## 7. Visualiser les clusters avec UMAP

Appliquer UMAP avec les scores d'urgence comme feature pondérée supplémentaire pour améliorer la séparation des clusters.

In [ ]:
# Appliquer UMAP avec poids d'urgence pour meilleure séparation des clusters
print("⏳ Application d'UMAP avec poids d'urgence...")

# Augmenter les données avec les scores d'urgence (poids 10x pour rapprocher les clusters)
urgency_feature = urgency_scores.reshape(-1, 1) * 10
augmented_data = np.hstack([reduced_data_scaled, urgency_feature])

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.15, metric='euclidean')
data_2d = reducer.fit_transform(augmented_data)

print(f"✓ Projection UMAP terminée")

# Créer la visualisation
fig, ax = plt.subplots(figsize=(16, 10))

# Définir les couleurs pour les niveaux d'urgence
couleurs_urgence = {
    0: "#90EE90",    # Bénin - Vert clair
    1: "#90EE90",    # Faible - Vert
    2: "#FFD700",    # Modéré - Or
    3: "#FF8C00",    # Grave - Orange foncé
    4: "#FF0000"     # Urgent - Rouge
}

labels_urgence = {
    0: "🟢 Bénin",
    1: "🟢 Faible",
    2: "🟡 Modéré",
    3: "🟠 Grave",
    4: "🔴 Urgent"
}

# Tracer les points colorés par cluster
colors = [couleurs_urgence[label] for label in labels]
scatter = ax.scatter(data_2d[:, 0], data_2d[:, 1], c=colors, s=100, alpha=0.6, edgecolors='darkgray', linewidth=0.5)

# Ajouter les centroïdes des clusters avec labels
for cluster_id in range(5):
    cluster_mask = labels == cluster_id
    if cluster_mask.sum() > 0:
        centroid_x = data_2d[cluster_mask, 0].mean()
        centroid_y = data_2d[cluster_mask, 1].mean()
        
        # Obtenir les termes principaux pour ce cluster
        cluster_tfidf = tfidf_matrix[cluster_mask].mean(axis=0).A1
        top_indices = cluster_tfidf.argsort()[-3:][::-1]
        top_terms = [feature_names[i] for i in top_indices]
        
        ax.annotate(
            f"C{cluster_id}\n{labels_urgence[cluster_id]}\n{' / '.join(top_terms)}",
            xy=(centroid_x, centroid_y),
            fontsize=9,
            fontweight='bold',
            ha='center',
            va='center',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=couleurs_urgence[cluster_id], 
                     alpha=0.85, edgecolor='black', linewidth=2),
            zorder=10,
            color='black'
        )

ax.set_xlabel("Dimension UMAP 1", fontsize=12, fontweight='bold')
ax.set_ylabel("Dimension UMAP 2", fontsize=12, fontweight='bold')
ax.set_title("Clustering d'urgence médicale - Visualisation UMAP\n(Bénin, Faible, Modéré, Grave, Urgent)", 
            fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.25, linestyle='--')

plt.tight_layout()
plt.show()

## 8. Créer un graphique de distribution des clusters

In [ ]:
# Créer le graphique en barres de distribution des clusters
fig, ax = plt.subplots(figsize=(12, 6))

ids_clusters = []
comptages = []
couleurs_liste = []

for cluster_id in range(5):
    count = np.sum(labels == cluster_id)
    nom_niveau = labels_urgence[cluster_id]
    
    ids_clusters.append(f"{nom_niveau}\nC{cluster_id}")
    comptages.append(count)
    couleurs_liste.append(couleurs_urgence[cluster_id])

bars = ax.bar(range(len(ids_clusters)), comptages, color=couleurs_liste, edgecolor='black', alpha=0.8, linewidth=2)

# Ajouter les labels de comptage sur les barres
for bar, count in zip(bars, comptages):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(count)}\n({count/len(texts)*100:.1f}%)',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_xlabel("Clusters de niveaux d'urgence", fontsize=12, fontweight='bold')
ax.set_ylabel("Nombre d'échantillons", fontsize=12, fontweight='bold')
ax.set_title("Distribution d'urgence médicale - 5 niveaux", fontsize=14, fontweight='bold')
ax.set_xticks(range(len(ids_clusters)))
ax.set_xticklabels(ids_clusters, fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, max(comptages) * 1.15)

plt.tight_layout()
plt.show()

## 9. Analyser et afficher les détails des clusters

In [1]:
# Analyser chaque cluster en détail
print("=" * 80)
print("ANALYSE DÉTAILLÉE DES CLUSTERS")
print("=" * 80)

for cluster_id in range(5):
    cluster_mask = labels == cluster_id
    cluster_texts = [texts[i] for i, mask in enumerate(cluster_mask) if mask]
    cluster_scores = urgency_scores[cluster_mask]
    
    # Obtenir les termes TF-IDF principaux
    cluster_tfidf = tfidf_matrix[cluster_mask].mean(axis=0).A1
    top_indices = cluster_tfidf.argsort()[-5:][::-1]
    top_terms = [(feature_names[i], cluster_tfidf[i]) for i in top_indices]
    
    count = len(cluster_texts)
    avg_score = cluster_scores.mean()
    
    print(f"\n{'─' * 80}")
    print(f"Cluster {cluster_id} : {labels_urgence[cluster_id]}")
    print(f"{'─' * 80}")
    print(f"  📊 Statistiques :")
    print(f"     - Échantillons : {count}")
    print(f"     - Pourcentage : {(count/len(texts))*100:.1f}%")
    print(f"     - Score d'urgence moyen : {avg_score:.2f}/5.0")
    print(f"     - Plage de scores : [{cluster_scores.min():.2f}, {cluster_scores.max():.2f}]")
    
    print(f"\n  🏆 Top 5 termes (par TF-IDF) :")
    for rank, (term, score) in enumerate(top_terms, 1):
        print(f"     {rank}. {term:25s} ({score:.4f})")
    
    print(f"\n  📝 Exemples de témoignages :")
    for idx, doc in enumerate(cluster_texts[:2], 1):
        preview = doc[:100] + "..." if len(doc) > 100 else doc
        print(f"     [{idx}] {preview}")

print(f"\n{'=' * 80}")

ANALYSE DÉTAILLÉE DES CLUSTERS


NameError: name 'labels' is not defined

## 10. Sauvegarder les résultats en CSV

In [ ]:
# Sauvegarder les résultats en CSV
CHEMIN_RESULTATS = os.path.join("results", "cluster_labels_urgency.csv")
os.makedirs(os.path.dirname(CHEMIN_RESULTATS), exist_ok=True)

print("📝 Sauvegarde des résultats du clustering en CSV...")

with open(CHEMIN_RESULTATS, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "cluster", "niveau_urgence", "score_urgence"])
    
    for id_, cluster, score in zip(ids, labels, urgency_scores):
        nom_niveau = labels_urgence[cluster].split()[1]  # Extraire le nom de niveau sans emoji
        writer.writerow([id_, cluster, nom_niveau, f"{score:.4f}"])

print(f"\n✓ Résultats sauvegardés dans : {CHEMIN_RESULTATS}")

# Afficher le résumé
print(f"\n" + "=" * 80)
print("✓ CLUSTERING TERMINÉ !")
print("=" * 80)
print(f"\n📊 Statistiques de résumé :")
print(f"   - Total d'échantillons : {len(texts)}")
print(f"   - Nombre de clusters : 5")
print(f"   - Attribution des clusters : Basée sur les scores (Bénin < 1.8 < Faible < 2.6 < Modéré < 3.4 < Grave < 4.2 < Urgent)")
print(f"\n✅ Sorties :")
print(f"   - Fichier CSV : {CHEMIN_RESULTATS}")
print(f"   - Visualisations : Graphique UMAP scatter + Graphique en barres de distribution")
print(f"\n🎯 Caractéristiques de l'algorithme :")
print(f"   - Scoring pondéré des mots-clés (5 niveaux d'urgence)")
print(f"   - Vectorisation TF-IDF (100 features, bigrammes)")
print(f"   - Réduction dimensionnelle PCA (variance expliquée calculée)")
print(f"   - Visualisation UMAP avec augmentation de poids d'urgence")
print(f"\n" + "=" * 80)